In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path(".").resolve()))

from scenarios.utils import set_seed, load_real_images, load_synth_images
from scenarios.generators import generate_deficit_fill, generate_total_mix

REAL_TRAIN_DIR = r"../train"
SYNTH_OK_DIR = r"../synthetic_images"
OUTPUT_ROOT = r"."

DEFICIT_PROPORTIONS = [0.25, 0.50, 0.75, 1.00]
TOTAL_MIX_PROPORTIONS = [0.25, 0.50, 0.75, 1.00]
SEED = 42


def generar_escenarios() -> None:
    set_seed(SEED)
    real_root = Path(REAL_TRAIN_DIR)
    synth_root = Path(SYNTH_OK_DIR)
    output_root = Path(OUTPUT_ROOT)

    print(f"\n{'='*60}")
    print("LECTURA DE FUENTES")
    print(f"{'='*60}")

    real_by_class = load_real_images(real_root)
    synth_by_class = load_synth_images(synth_root)

    generate_deficit_fill(
        real_by_class=real_by_class,
        synth_by_class=synth_by_class,
        output_root=output_root,
        proportions=DEFICIT_PROPORTIONS,
    )

    generate_total_mix(
        real_by_class=real_by_class,
        synth_by_class=synth_by_class,
        output_root=output_root,
        proportions=TOTAL_MIX_PROPORTIONS,
    )

    print(f"\n{'='*60}")
    print("Escenarios generados")
    print(f"{'='*60}")


# generar_escenarios()

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def plot_stacked_distributions(summary_path):
    path_obj = Path(summary_path)
    if not path_obj.exists():
        print(f"No existe: {summary_path}. Ejecuta generar_escenarios() primero.")
        return

    with open(path_obj, "r", encoding="utf-8") as f:
        data = json.load(f)

    classes = sorted(data["classes"].keys(), key=lambda x: int(x))
    real_counts = np.array([data["classes"][c]["n_real"] for c in classes])
    synth_counts = np.array([data["classes"][c]["n_synth"] for c in classes])

    plt.rcParams.update({"font.size": 14})
    fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

    x = np.arange(len(classes))
    labels = [str(c) for c in classes]

    ax.bar(x, real_counts, 0.8, label="Originales", color="#1f77b4")
    ax.bar(x, synth_counts, 0.8, bottom=real_counts, label="Sinteticas", color="#ff7f0e")

    ax.set_ylabel("Numero de imagenes")
    ax.set_xlabel("Clases")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend(loc="upper right")

    scenario = data.get("scenario", "")
    mode = data.get("mode", "")
    total = int(np.sum(real_counts) + np.sum(synth_counts))
    ax.set_title(f"Distribucion por clase ({mode} - {scenario})\nTotal: {total}", pad=15)

    plt.tight_layout()
    plt.show()


print("plot_stacked_distributions('train_deficit/train_50/summary.json')")
print("plot_stacked_distributions('train_total/train_75/summary.json')")

In [ ]:
for p in DEFICIT_PROPORTIONS:
    perc = int(round(p * 100))
    plot_stacked_distributions(f"train_deficit/train_{perc}/summary.json")

for p in TOTAL_MIX_PROPORTIONS:
    perc = int(round(p * 100))
    plot_stacked_distributions(f"train_total/train_{perc}/summary.json")